<a href="https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
# Clone the repository to access necessary files and modules
repo_name = "flyrank-ml-internship-shwetabh"
repo_url = f"https://github.com/Shwetabh1013/{repo_name}.git"

# Check if the repository already exists to avoid re-cloning
if not Path(f"/content/{repo_name}").is_dir():
    print(f"Cloning {repo_url} into /content/{repo_name}...")
    !git clone {repo_url} /content/{repo_name}
    print("Repository cloned successfully.")
else:
    print(f"Repository already exists at /content/{repo_name}. Skipping clone.")

# Change to the notebook's expected directory within the cloned repo
# This ensures relative paths (like 'data/raw/') work as expected
notebook_dir = Path(f"/content/{repo_name}/work/notebooks")
if notebook_dir.is_dir():
    os.chdir(notebook_dir)
    print(f"Changed current working directory to: {os.getcwd()}")
else:
    print(f"Warning: Notebook directory {notebook_dir} not found. Current working directory remains: {os.getcwd()}")

Repository already exists at /content/flyrank-ml-internship-shwetabh. Skipping clone.
Changed current working directory to: /content/flyrank-ml-internship-shwetabh/work/notebooks


## 1. Method choice and why

**Question shape:** yes/no with an observed label — `is_declining_label` (1 when `trend_direction == "down"`, ~54% of rows). Per the toolkit, that shape starts with **Logistic Regression** (readable, gives coefficients I can sanity-check), then **Random Forest** (captures interactions my hand-built W04 rule had to *guess* at manually — the rule hard-codes "stale AND CTR-gap" as one fixed AND condition; a tree model can find that kind of interaction, and others I didn't think to encode, on its own).

I'm skipping Gradient Boosting and clustering here: this is a supervised yes/no task, not an unsupervised "group these" or open-ended "what drives X" question, so clustering doesn't fit. A few points of AUC from boosting isn't worth losing the readable coefficients and split-level feature importances I get from the two simpler models — simplicity is a feature at this stage, not a shortcut.

**Banned as inputs** (confirmed against `docs/data-dictionary.md` and my own W04 leakage check): `trend_direction` / `trend_pct` (the label's source) and `is_declining_label` itself (the target).

In [11]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# The repository is cloned into /content/<repo_name>
repo_name = "flyrank-ml-internship-shwetabh"
repo_root = Path(f"/content/{repo_name}")

# Define absolute paths for the scripts directory and ml_utils file
scripts_path = repo_root / "scripts"
ml_utils_file = scripts_path / "ml_utils.py"

# Define the absolute path for the data file
data_file_path = repo_root / "data/raw/content_refresh_anonymized.csv"

# Add the scripts directory to sys.path and import ml_utils
if scripts_path.is_dir() and ml_utils_file.is_file():
    sys.path.insert(0, str(scripts_path))
    from ml_utils import precision_at_k
    print(f"Successfully added '{scripts_path}' to sys.path and imported ml_utils.")
elif not scripts_path.is_dir():
    raise ModuleNotFoundError(f"Error: The 'scripts' directory was not found at '{scripts_path}'. Please ensure the repository is cloned correctly.")
elif not ml_utils_file.is_file():
    raise ModuleNotFoundError(f"Error: The 'ml_utils.py' file was not found inside '{scripts_path}'. Please check the file's existence and spelling.")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load the data using its absolute path
if data_file_path.is_file():
    df = pd.read_csv(data_file_path)
else:
    raise FileNotFoundError(f"Error: Data file not found at '{data_file_path}'. Please ensure the repository is cloned correctly and the file exists.")

# same label definition as docs/data-dictionary.md: is_declining_label = (trend_direction == "down")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"{len(df):,} rows, {df['client_id'].nunique()} clients, "
      f"declining rate: {df['is_declining_label'].mean():.3f}")

Successfully added '/content/flyrank-ml-internship-shwetabh/scripts' to sys.path and imported ml_utils.
30,000 rows, 32 clients, declining rate: 0.542


## 2. Split design

**Grouped by client, not a random row split.** `client_id` is a pseudonym for grouping/joins only, per the data dictionary — never a feature. But a random row split would let the model see some of a client's pages in training and then recognize that *same client's* other pages in the test set, through correlated traits (a client's typical `content_type` mix, general traffic level, etc.) rather than learning something that actually generalizes across clients.

32 distinct clients. I'm holding out ~20% of **clients** entirely (not rows) for testing — the same idea as `scripts/03_train_model.py`'s `client_holdout`, reproduced directly here so the split logic is visible in this notebook without needing to open that file. Fixed seed (42), so this is reproducible.

In [12]:
client_ids = df["client_id"].unique()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(client_ids)
n_test_clients = max(1, round(len(shuffled) * 0.2))
test_clients = set(shuffled[:n_test_clients])

test_mask = df["client_id"].isin(test_clients)
train_df = df[~test_mask].reset_index(drop=True)
test_df = df[test_mask].reset_index(drop=True)

print(f"train: {len(train_df):,} rows / {train_df['client_id'].nunique()} clients")
print(f"test:  {len(test_df):,} rows / {test_df['client_id'].nunique()} clients")
print(f"train declining rate: {train_df['is_declining_label'].mean():.3f}")
print(f"test declining rate:  {test_df['is_declining_label'].mean():.3f}")

assert set(train_df["client_id"]).isdisjoint(set(test_df["client_id"])), \
    "client leakage across the split!"

train: 27,675 rows / 26 clients
test:  2,325 rows / 6 clients
train declining rate: 0.555
test declining rate:  0.391


In [17]:
display(train_df.head())

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [18]:
# Check for missing values in train_df
print("Missing values in train_df:")
display(train_df.isnull().sum()[train_df.isnull().sum() > 0].sort_values(ascending=False))

Missing values in train_df:


,0
provider_used,19998
word_count_tier,7609
char_count,7609
word_count,7609
char_count_tier,7609
model_used,5733
trend_pct,2562
competition_level,1565
search_volume,1423
competition,1423


In [14]:
# Check for missing values in train_df
print("Missing values in train_df:")
display(train_df.isnull().sum()[train_df.isnull().sum() > 0].sort_values(ascending=False))

Missing values in train_df:


,0
provider_used,19998
word_count_tier,7609
char_count,7609
word_count,7609
char_count_tier,7609
model_used,5733
trend_pct,2562
competition_level,1565
search_volume,1423
competition,1423


## 3. Train + compare vs my baseline

**My W04 rule, recomputed here on this split** — same logic as `w04_baseline_score.ipynb`: a page is `stale_and_ctr_underperform` when `days_since_last_update >= 91` AND its `ctr` is under half its `position_tier`'s weighted CTR benchmark, scored by `impressions_90d`.

**One honest change from the original W04 version:** there, the CTR benchmark was computed over the *whole* dataset. Here I compute it from **train-client rows only**, then apply it to the test rows — otherwise the baseline would be peeking at test-set statistics that the model never gets to see, which would make the comparison unfair in the baseline's favor.

**Metric: precision@K** (K=20, K=50), same metric family as the `training-honest-models` skill and the repo's own `ml_utils.precision_at_k`. This is a ranking/queue problem, not a threshold classification problem, so precision@K — "how good is the top of the queue" — is the metric that actually matches the decision editors make.

In [15]:
# --- Recompute my W04 rule, fit only on train, applied to test ---
position_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

train_tier_ctr = (
    train_df[train_df["position_tier"].isin(position_order)]
    .groupby("position_tier")
    .apply(lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100)
)

def score_baseline(frame):
    tier_benchmark = frame["position_tier"].map(train_tier_ctr)
    stale = (frame["days_since_last_update"] >= 91).astype(int)
    ctr_gap = (
        (frame["ctr"] < 0.5 * tier_benchmark) & frame["position_tier"].isin(position_order)
    ).astype(int)
    return stale * ctr_gap * frame["impressions_90d"]

test_df = test_df.copy()
test_df["baseline_score"] = score_baseline(test_df).fillna(0)

baseline_p20 = precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 20)
baseline_p50 = precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 50)
print(f"W04 rule on test clients -- precision@20: {baseline_p20:.3f}, precision@50: {baseline_p50:.3f}")

# --- Features: safe list only, no label-source or product-flag columns ---
NUMERIC_FEATURES = [
    "days_since_last_update", "ctr", "avg_position", "impressions_90d", "clicks_90d",
    "engagement_rate", "scroll_rate", "content_age_days", "word_count",
]
CATEGORICAL_FEATURES = ["position_tier", "freshness_tier", "content_type", "main_intent"]

def build_features(frame):
    numeric = frame[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
    categorical = pd.get_dummies(
        frame[CATEGORICAL_FEATURES].fillna("unknown").astype(str),
        prefix=CATEGORICAL_FEATURES,
    )
    return pd.concat([numeric.reset_index(drop=True), categorical.reset_index(drop=True)], axis=1)

X_train = build_features(train_df)
y_train = train_df["is_declining_label"]
X_test = build_features(test_df).reindex(columns=X_train.columns, fill_value=0)
y_test = test_df["is_declining_label"]

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

results = {
    "W04_baseline_rule": {"precision_at_20": baseline_p20, "precision_at_50": baseline_p50, "roc_auc": None},
}
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "precision_at_20": precision_at_k(y_test, proba, 20),
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "roc_auc": roc_auc_score(y_test, proba),
    }

base_rate = y_test.mean()
results["base_rate_random_ranking"] = {
    "precision_at_20": base_rate, "precision_at_50": base_rate, "roc_auc": 0.5,
}

comparison_table = pd.DataFrame(results).T.round(3)
comparison_table

W04 rule on test clients -- precision@20: 0.250, precision@50: 0.180


/tmp/ipykernel_1782/3616277394.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100)
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,precision_at_20,precision_at_50,roc_auc
W04_baseline_rule,0.250,0.180,NaN
logistic_regression,0.800,0.640,0.691
random_forest,0.550,0.620,0.732
base_rate_random_ranking,0.391,0.391,0.500


## 4. Errors and interpretation

*Fill in the two bracketed lines below after running the cell -- they depend on the actual numbers, which only exist once this notebook has executed.*

- **Did any model beat the W04 rule?** [TODO: name the model and the precision@20/@50 delta from the table above. If nothing beat it, say that plainly -- same discipline as keeping the W02 result instead of tuning it away.]
- **What does the winning approach lean on?** [TODO: name the top 2-3 features from the importance table below, and say in one sentence each whether that makes sense or looks suspiciously perfect -- suspiciously perfect usually means leakage, not skill.]

Read below before believing the score.

In [16]:
best_name = max(("logistic_regression", "random_forest"),
                 key=lambda n: results[n]["precision_at_50"])
best_model = fitted[best_name]
print(f"Best model by precision@50: {best_name}")
print()

# --- Feature importance / coefficients, to sanity-check what the model leans on ---
if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(best_model.feature_importances_, index=X_train.columns)
else:
    importance = pd.Series(best_model.coef_[0], index=X_train.columns).abs()
importance = importance.sort_values(ascending=False)
print("Top 10 features:")
print(importance.head(10))
print()

# --- Where is it most wrong? ---
test_df["predicted_proba"] = best_model.predict_proba(X_test)[:, 1]
test_df["predicted_label"] = (test_df["predicted_proba"] >= 0.5).astype(int)
test_df["error_type"] = np.select(
    [
        (test_df["is_declining_label"] == 1) & (test_df["predicted_label"] == 0),
        (test_df["is_declining_label"] == 0) & (test_df["predicted_label"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)
print(test_df["error_type"].value_counts())
print()
print("Error rate by position_tier:")
print(test_df.groupby("position_tier")["error_type"].apply(lambda s: (s != "correct").mean()).round(3))
print()

# --- 3 concrete wrong cases, ranked by how confidently the model got them wrong ---
wrong = test_df[test_df["error_type"] != "correct"].copy()
wrong["confidence_gap"] = (wrong["predicted_proba"] - 0.5).abs()
wrong = wrong.sort_values("confidence_gap", ascending=False)

cols_to_show = [
    "content_id", "position_tier", "ctr", "days_since_last_update",
    "impressions_90d", "trend_direction", "predicted_proba", "error_type",
]
wrong[cols_to_show].head(3)

# TODO after running: add a markdown cell below with one sentence per case explaining
# *why* each one is hard -- e.g. a page with strong signals pointing one way but the
# opposite actual outcome. Same spirit as the W04 top-10 "what would make it wrong" review.

Best model by precision@50: logistic_regression

Top 10 features:
ctr                             0.346821
position_tier_top_3             0.244221
content_type_keyword article    0.205516
position_tier_striking          0.178722
position_tier_page_1            0.142315
main_intent_informational       0.127098
content_type_feedly article     0.118344
main_intent_unknown             0.107510
freshness_tier_0-30             0.089898
position_tier_page_3_5          0.071573
dtype: float64

error_type
correct           1516
false_positive     516
false_negative     293
Name: count, dtype: int64

Error rate by position_tier:
position_tier
deep        0.400
page_1      0.394
page_3_5    0.399
striking    0.432
top_3       0.154
Name: error_type, dtype: float64



,content_id,position_tier,ctr,days_since_last_update,impressions_90d,trend_direction,predicted_proba,error_type
1970,content_a8cee66e4788,top_3,100.0,20,1,down,3.654344e-16,false_negative
2131,content_e454093819d5,top_3,75.0,211,4,down,5.911954e-12,false_negative
1790,content_c0af3d6f9dd3,page_1,50.0,20,2,down,1.801456e-08,false_negative


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.